# Error analysis with Poniard

Beyond a leaderboard, Poniard keeps going into *where and why* your models fail. `ErrorAnalyzer` ranks the worst samples and cross-tabulates them against the target and the features.

In [ ]:
from sklearn.datasets import make_classification
from poniard import PoniardClassifier
from poniard.error_analysis import ErrorAnalyzer

X, y = make_classification(
    n_samples=200, n_features=10, n_informative=5, random_state=42
)
clf = PoniardClassifier()
clf.fit(X, y)

In [ ]:
ea = ErrorAnalyzer.from_poniard(
    clf, estimator_names=["LogisticRegression", "RandomForestClassifier"]
)

## One call: the full report

`analyze` runs the whole workflow and packages it into a single report.

In [ ]:
report = ea.analyze(X, y)
list(report)

In [ ]:
# per-estimator: how many samples failed and the error rate
report["summary"]

In [ ]:
# per sample: how many estimators failed on it, and their average error
report["merged_errors"]

## The individual steps

Rank the worst samples per estimator, then merge across estimators.

In [ ]:
ranked = ea.rank_errors(X, y)
# worst samples for the RandomForest, ranked by how confidently wrong it is
ranked["RandomForestClassifier"].head(10)

In [ ]:
merged = ErrorAnalyzer.merge_errors(ranked)
merged.head(10)

## Where do the errors live?

Errors per target class, including the error rate (the share of each class that is misclassified).

In [ ]:
ea.analyze_target(errors_idx=merged.index, y=y)

In [ ]:
# per feature: numeric summaries by error group, error rate per category
ea.analyze_features(errors_idx=merged.index, X=X)

## Feature-level analysis with permutation importance

Restrict to the top features by permutation importance (computed on the same cross-validation folds Poniard used).

In [ ]:
ea.analyze_features(
    errors_idx=merged.index, X=X, y=y,
    estimator_name="LogisticRegression", n_features=3,
)